# MLP Test

## Notes
- For use when test set is available

## File Conversion

### Imports

In [1]:
# General Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from math import ceil
from time import perf_counter
from datetime import datetime

In [ ]:
# TODO: Load test ECG
assert Path('train_ecg.npz').is_file(), "test_ecg.npz was not found"
TRAIN_ECG  = np.load('train_ecg.npz')
TRAIN_ECGx = TRAIN_ECG['X']
TRAIN_ECGy = TRAIN_ECG['y']

assert Path('test_ecg.npz').is_file(), "test_ecg.npz was not found"
TEST_ECG  = np.load('test_ecg.npz')
TEST_ECGx = TEST_ECG['X']
TEST_ECGy = TEST_ECG['y']

df_train = pd.DataFrame(TRAIN_ECGx, columns=[f'x_{i+1}' for i in range(TRAIN_ECGx.shape[1])])
df_train['target'] = TRAIN_ECGy

df_test = pd.DataFrame(TEST_ECGx, columns=[f'x_{i+1}' for i in range(TEST_ECGx.shape[1])])
df_test['target'] = TEST_ECGy

AssertionError: test_ecg.npz was not found

In [ ]:
{
    'df_train': df_train.shape,
    'df_test': df_test.shape
}

NameError: name 'df_test' is not defined

In [ ]:
df_train['target'].value_counts()

In [ ]:
df_test['target'].value_counts()

## MLP NN

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

In [ ]:
print('CUDA:', torch.version.cuda, '| Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0))

CUDA: 13.4 | Available: True | Device: NVIDIA GeForce RTX 4070 Laptop GPU


In [ ]:
SEED = 825
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) # Set the random seed for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Set the device to GPU if available, otherwise CPU

# Class wights for training set frequencies
class_counts = np.bincount(TRAIN_ECGy)
class_weights = len(TRAIN_ECGy) / (len(class_counts) * class_counts)
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
    ).to(device)
print(f"Class weights: \n{class_weights}")

criterion      = nn.CrossEntropyLoss(weight=class_weights) # Loss Function for multi-class classification
eval_criterion = nn.CrossEntropyLoss()

Class weights: 
tensor([0.5516, 5.3482], device='cuda:0')


In [ ]:
feature_cols = [c for c in df_train.columns if c != 'target']
scaler = StandardScaler().fit(df_train[feature_cols])

def to_tensor_dataset(df):
    '''
    Substitute to **W11_D2**'s `balanced_subset()` function. \n
    Converts a DataFrame into a PyTorch TensorDataset.
    '''
    # Extract features and labels
    X = torch.tensor(scaler.transform(df[feature_cols]), dtype=torch.float32)
    y = torch.tensor(df['target'].values, dtype=torch.long)
    
    return TensorDataset(X, y)

In [ ]:
test_data = to_tensor_dataset(df_test)

test_loader = DataLoader(test_data, batch_size=512, shuffle=False)

In [ ]:
print(f"Test DataLoader: {len(test_loader)} batches, each with {test_loader.batch_size} samples")

### `evaluate(model, loader)` Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, probs, labels = 0.0, [], []
    with torch.inference_mode():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            total += eval_criterion(logits, y_batch).item() * len(y_batch) # criterion(logits, y_batch).item() * len(y_batch)
            probs.append(torch.softmax(logits, dim=1)[:, 1].cpu())  # P(abnormal)
            labels.append(y_batch.cpu())
    return total / len(loader.dataset), torch.cat(probs).numpy(), torch.cat(labels).numpy()

## Retrieve Parameters

In [9]:
import json

parameters_path = Path("contingency_parameters.txt")
assert parameters_path.is_file(), f"{parameters_path} was not found"

parameters_text = (parameters_path.read_text())
parameters = json.loads(parameters_text)
parameters

{'Batch Size': 5119,
 'Epochs': 100,
 'Alpha': 0.01,
 'Weight Decay': 0.001,
 'TP / FN Ratio': 34.9756}

### `fit(model_class)` Function

In [ ]:
def fit(
        model_class, 
        data,
        ITER_epochs,
        ITER_batchSize,
        ITER_alpha,
        ITER_weight_decay):
    torch.manual_seed(SEED)
    
    # Create instance of model
    model = model_class().to(device)
    loader = DataLoader(
        data, 
        batch_size=ITER_batchSize, # Full Batch
        shuffle   =False
    )
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr          =ITER_alpha,
        weight_decay=ITER_weight_decay)

    epoch_log, loss_log, = [], []
    for epoch in range(ITER_epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in loader:
            epoch_start = perf_counter() # Start timer
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Backpropigation
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        loss_proportion = total_loss / len(loader)
        epoch_elapsed = perf_counter() - epoch_start # End timer
        print(f"Epoch {epoch+1}/{ITER_epochs}, Loss: {loss_proportion:.4f} | {epoch_elapsed:.3f}s")
        epoch_log.append(epoch+1)
        loss_log.append(loss_proportion)
    return model, epoch_log, loss_log

### MLP Class

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()       # Flattens the input tensor to a 1D tensor
        self.hidden  = nn.Linear(252, 32) # Mapping 252 input features (x_n) to 32 hidden neurons
        self.output  = nn.Linear(32, 2)   # Mapping 32 hidden neurons to 2 output classes (normal/abnormal)
    
    def forward(self, x):
        z = self.hidden(self.flatten(x))
        h = torch.relu(z)
        return self.output(h)

## Finalized
Aims to isolate and rerun the model with the best TP:FN ratio.
### Training

In [ ]:
MLP_MODEL_Final, MLP_MODEL_epoch_log_Final, MLP_MODEL_loss_log_Final = fit(
    MLP, train_data,
    ITER_epochs      =final_parameters['Epochs'],      # 100
    ITER_batchSize   =final_parameters['Batch Size'],  # 5119 (full)
    ITER_alpha       =final_parameters['Alpha'],       # 0.01
    ITER_weight_decay=final_parameters['Weight Decay'] # 0.001
)

Epoch 1/100, Loss: 1.0846 | 0.006s
Epoch 2/100, Loss: 0.3822 | 0.005s
Epoch 3/100, Loss: 0.3612 | 0.012s
Epoch 4/100, Loss: 0.2311 | 0.006s
Epoch 5/100, Loss: 0.2344 | 0.012s
Epoch 6/100, Loss: 0.1683 | 0.006s
Epoch 7/100, Loss: 0.1403 | 0.004s
Epoch 8/100, Loss: 0.1203 | 0.004s
Epoch 9/100, Loss: 0.1095 | 0.004s
Epoch 10/100, Loss: 0.1036 | 0.004s
Epoch 11/100, Loss: 0.0943 | 0.006s
Epoch 12/100, Loss: 0.0872 | 0.004s
Epoch 13/100, Loss: 0.0807 | 0.004s
Epoch 14/100, Loss: 0.0751 | 0.004s
Epoch 15/100, Loss: 0.0700 | 0.013s
Epoch 16/100, Loss: 0.0657 | 0.010s
Epoch 17/100, Loss: 0.0611 | 0.011s
Epoch 18/100, Loss: 0.0580 | 0.005s
Epoch 19/100, Loss: 0.0532 | 0.013s
Epoch 20/100, Loss: 0.0519 | 0.005s
Epoch 21/100, Loss: 0.0465 | 0.004s
Epoch 22/100, Loss: 0.0446 | 0.003s
Epoch 23/100, Loss: 0.0420 | 0.003s
Epoch 24/100, Loss: 0.0395 | 0.005s
Epoch 25/100, Loss: 0.0372 | 0.013s
Epoch 26/100, Loss: 0.0361 | 0.005s
Epoch 27/100, Loss: 0.0344 | 0.011s
Epoch 28/100, Loss: 0.0340 | 0.006s
E